# 05 · Currency, conversion, imputation and the monthly grid (Mexico)
**Country series · Mexico** · *generated from `notebooks/country_template`: edit the template*

Conformed silver converts every amount to USD, fills missing USD amounts with the daily rate, assumes a direction
for money, converts incomes and builds a dense monthly grid (pipeline series, notebook 05). Each of those decisions
is re-examined here on Mexico's currencies.

**Rule for this country: analyse in the account's currency, compare in USD.** Statistics such as medians, outlier
thresholds and structuring bands are computed per currency; only cross-country comparisons use USD.

In [1]:
import sys
import time

sys.path.insert(0, "../../src")
import plotly.express as px
from IPython.display import Markdown, display
from itables import show

from latam_eda import country, theme

COUNTRY = "MX"
CTRY = country.COUNTRIES[COUNTRY]
theme.register()
t0 = time.time()
pl = country.session(COUNTRY)
country.prepare(pl, "seeds")
CONFORMED = [
    "int_fx_usd",
    "int_customer_profile",
    "int_customer_fairness_attributes",
    "int_products_enriched",
    "int_transactions_enriched",
    "int_customer_month_tx",
    "int_transactions_enriched_with_holdout",
]
built = pl.build_set(CONFORMED)

  table     int_customer_fairness_attributes                      74,907 rows     0.1s
  table     int_fx_usd                                             4,388 rows     0.0s
  table     int_customer_profile                                  74,907 rows     0.4s
  table     int_transactions_enriched                          2,150,940 rows     3.7s
  table     int_customer_month_tx                              2,696,652 rows     0.8s
  table     int_products_enriched                                200,398 rows     0.5s
  table     int_transactions_enriched_with_holdout             2,216,431 rows     7.0s


## 1 · The country's exchange rate

In [2]:
fx = pl.q(f"""select rate_date, usd_per_unit, 1 / usd_per_unit as units_per_usd
              from {{int_fx_usd}} where currency = '{CTRY.home_currency}' order by 1""")
fig = px.line(fx, x="rate_date", y="units_per_usd", title=f"{CTRY.home_currency} per USD")
fig.update_layout(height=300, xaxis_title=None)
fig.show()
move = 100 * (fx.units_per_usd.iloc[-1] / fx.units_per_usd.iloc[0] - 1)
cv = 100 * fx.units_per_usd.std() / fx.units_per_usd.mean()
real = {
    "MX": "the peso moved between about 16.5 and 20.5 per dollar (2023–2025)",
    "CO": "the peso moved between about 3,800 and 4,500 per dollar (2023–2025)",
    "AR": "the official peso went from about 260 to over 1,000 per dollar (2023–2025), including a devaluation of "
    "more than 50 % in December 2023",
}
display(
    Markdown(
        f"**{CTRY.home_currency} per USD moves {move:+.1f} % over three years (coefficient of variation {cv:.1f} %).** "
        f"In reality {real[COUNTRY]}. Every USD figure in this country is therefore "
        "stable by construction; any feature meant to capture devaluation, inflation or real income cannot be validated "
        "here."
        + (
            " For Argentina this matters most: a real Argentine model must work in inflation-adjusted terms."
            if COUNTRY == "AR"
            else ""
        )
    )
)

**MXN per USD moves +0.9 % over three years (coefficient of variation 1.2 %).** In reality the peso moved between about 16.5 and 20.5 per dollar (2023–2025). Every USD figure in this country is therefore stable by construction; any feature meant to capture devaluation, inflation or real income cannot be validated here.

## 2 · Amounts per currency

In [3]:
amt = pl.q("""select currency, count(*) as transactions, round(median(amount), 2) as median_amount,
                     round(quantile_cont(amount, 0.99), 2) as p99_amount, round(median(amount_usd), 2) as median_usd,
                     round(quantile_cont(amount_usd, 0.99), 2) as p99_usd
              from {int_transactions_enriched} group by 1 order by 2 desc""")
show(amt, paging=False)
dist = pl.q("""select currency, floor(log10(greatest(amount, 0.01)) * 4) / 4 as log10_amount_bin, count(*) as n_tx
               from {int_transactions_enriched} group by all order by 1, 2""")
fig = px.bar(
    dist,
    x="log10_amount_bin",
    y="n_tx",
    color="currency",
    barmode="overlay",
    title=f"{CTRY.name}: distribution of log10(amount) per currency",
)
fig.update_layout(height=320)
fig.show()
display(
    Markdown(
        "**In USD every currency has the same median (about 467): amounts were generated in dollars and converted.** "
        "Per-currency statistics are still the right default: a threshold in pesos and one in dollars cannot share a "
        "number, and a model fed raw `amount` across currencies would learn the currency, not the behaviour."
    )
)

**In USD every currency has the same median (about 467): amounts were generated in dollars and converted.** Per-currency statistics are still the right default: a threshold in pesos and one in dollars cannot share a number, and a model fed raw `amount` across currencies would learn the currency, not the behaviour.

## 3 · The USD imputation in this country
`amount_usd` is missing for some non-USD transactions and is filled with amount × the published daily rate
(`dq_r18_amount_usd_imputed`). Its error, measured where the source gives the value:

In [4]:
mix = pl.q("""
    select case when currency = 'USD' then 'USD, no conversion'
                when not dq_r18_amount_usd_imputed then 'source amount_usd' else 'imputed' end as origin,
           count(*) as transactions
    from {int_transactions_enriched} group by 1 order by 2 desc""")
mix["share %"] = (100 * mix.transactions / mix.transactions.sum()).round(2)
show(mix, paging=False)
err = pl.q("""
    with e as (select t.currency, 100 * (t.amount * f.usd_per_unit / t.amount_usd_source - 1) as rel_err_pct
               from {stg_transactions} t asof left join {int_fx_usd} f
                 on f.currency = t.currency and f.rate_date <= t.process_date
               where t.currency <> 'USD' and t.amount_usd_source is not null and t.amount <> 0)
    select currency, count(*) as n, round(avg(rel_err_pct), 3) as mean_pct, round(stddev(rel_err_pct), 3) as sd_pct,
           round(min(rel_err_pct), 2) as min_pct, round(max(rel_err_pct), 2) as max_pct
    from e group by 1""")
show(err, paging=False)
imp = mix.set_index("origin")["share %"].get("imputed", 0.0)
display(
    Markdown(
        (
            f"**{imp:.2f} % of {CTRY.name}'s transactions carry an imputed USD amount, with a uniform ±2 % error.** "
            if len(err)
            else f"**{CTRY.name} has no non-USD transactions: nothing is converted or imputed.** "
        )
        + (
            "Decision: keep the imputation (unbiased; dropping or zeroing would understate USD totals for exactly the "
            "rows missing the field) and keep the flag in every mart; treat imputed amounts within 2 % of any regulatory "
            "threshold as uncertain."
            if len(err)
            else "Every USD figure here is exact; the R18 rule cannot fire in this country."
        )
    )
)

**Mexico has no non-USD transactions: nothing is converted or imputed.** Every USD figure here is exact; the R18 rule cannot fire in this country.

## 4 · Incomes: the currency trap in reverse

In [5]:
inc = pl.q("""select home_currency, count(*) as customers, round(median(monthly_income_local), 0) as median_income_local,
                     round(median(monthly_income_usd), 0) as median_income_usd,
                     round(quantile_cont(monthly_income_usd, 0.9), 0) as p90_income_usd
              from {int_customer_profile} group by 1""")
show(inc, paging=False)
display(
    Markdown(
        f"Incomes are reported in **{CTRY.home_currency}**, converted with the {CTRY.home_currency} rate to a median of "
        f"about {inc.median_income_usd.iloc[0]:,.0f} USD a month. "
        + (
            "Mexican accounts are all in USD while incomes are in pesos: comparing an amount with an income without "
            "converting the income would make every Mexican look 17 times richer than they are. The special case in "
            "`int_customer_profile` is what keeps every income ratio right."
            if COUNTRY == "MX"
            else "Accounts mix the home currency and USD, so every income ratio must compare USD with USD."
        )
    )
)

Incomes are reported in **MXN**, converted with the MXN rate to a median of about 2,300 USD a month. Mexican accounts are all in USD while incomes are in pesos: comparing an amount with an income without converting the income would make every Mexican look 17 times richer than they are. The special case in `int_customer_profile` is what keeps every income ratio right.

## 5 · Direction and the monthly grid

In [6]:
show(
    pl.q("""select direction, transaction_type, count(*) as transactions,
                    round(100 * count(*) / sum(count(*)) over (), 2) as share_pct
             from {int_transactions_enriched} group by all order by 1, 2"""),
    paging=False,
)
grid = pl.q("""select count(*) as customer_months, round(100 * avg((n_tx = 0)::int), 2) as zero_month_pct,
                      round(100 * avg((inflow_usd > 0)::int), 2) as months_with_inflow_pct, round(avg(n_tx), 3) as avg_tx
               from {int_customer_month_tx}""")
show(grid, paging=False)
recent = pl.q("""select months_with_inflow, count(*) as customers from (
                   select customer_id, count(*) filter (where inflow_usd > 0) as months_with_inflow
                   from {int_customer_month_tx} where month_start >= date '2025-11-01' group by 1)
                 group by 1 order by 1""")
fig = px.bar(
    recent,
    x="months_with_inflow",
    y="customers",
    title="Months with any inflow, Nov 2025 – May 2026",
)
fig.update_layout(height=300)
fig.show()
reg = recent[recent.months_with_inflow >= 4].customers.sum() / recent.customers.sum()
display(
    Markdown(
        f"**{grid.zero_month_pct.iloc[0]:.1f} % of customer-months are empty and {100 * reg:.1f} % of customers had an "
        "inflow in four or more of the last seven months.** The zero-filled grid is the right imputation (no transaction "
        "is zero activity); the thinness is the data's, and it decides the credit outcomes of notebook 08."
    )
)

**51.3 % of customer-months are empty and 1.0 % of customers had an inflow in four or more of the last seven months.** The zero-filled grid is the right imputation (no transaction is zero activity); the thinness is the data's, and it decides the credit outcomes of notebook 08.

## 6 · Silver tests for the country

In [7]:
pl.ensure_until("silver")
tests = pl.run_tests("silver")
show(
    tests[tests.status != "pass"][["test", "attached_to", "severity", "failures", "status"]],
    paging=False,
)
dangling = tests[tests.test.str.contains("affected_product_id")]
if len(dangling) and dangling.failures.iloc[0]:
    display(
        Markdown(
            f"**New in the country cut: {int(dangling.failures.iloc[0]):,} complaints name a product that is not in "
            f"{CTRY.name}'s product table.** Bank-wide, every named product belonged to *another customer* (R25); the "
            "country cut shows most of those customers live in **another country**. The product link of a complaint is "
            "not just wrong, it crosses borders: no country team could ever have resolved it."
        )
    )

**New in the country cut: 10,808 complaints name a product that is not in Mexico's product table.** Bank-wide, every named product belonged to *another customer* (R25); the country cut shows most of those customers live in **another country**. The product link of a complaint is not just wrong, it crosses borders: no country team could ever have resolved it.

## Findings for Mexico and what to do
The computed statements above are the findings; the method holds for every country: per-currency statistics,
converted incomes, flagged imputations, a zero-filled grid, and the direction assumption stated in every model card.

In [8]:
pl.close()
print(f"notebook time {time.time() - t0:.0f}s")

notebook time 21s
